In [1]:
#-*- coding:UTF-8 -*-
import RPi.GPIO as GPIO
import time
import heapq
import threading
import cv2
import numpy as np
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
from aip import AipFace
import base64
import datetime
import socket
import os
import struct



In [2]:

# 小车电机引脚定义
IN1 = 20
IN2 = 21
IN3 = 19
IN4 = 26
ENA = 16
ENB = 13

# RGB三色灯引脚定义
LED_R = 22
LED_G = 27
LED_B = 24

# 舵机引脚定义
USPin = 10 #23
cameraV = 9
cameraH = 23 #11 10 4hao

# 超声波引脚定义
EchoPin = 0
TrigPin = 1

#蜂鸣器
BuzzerPin = 8

# 小车按键定义
key = 8

# 循迹红外引脚定义
TrackSensorLeftPin1 = 3
TrackSensorLeftPin2 = 5
TrackSensorRightPin1 = 4
TrackSensorRightPin2 = 18

# 设置GPIO口为BCM编码方式
GPIO.setmode(GPIO.BCM)

# 忽略警告信息
GPIO.setwarnings(False)

# 全局变量
pwm_ENA = None
pwm_ENB = None
pwm_servo_cameraH= None
pwm_servo_cameraV= None
pwm_servo_USPin= None
pwm_BUZZER_PIN = None
obstacle_detected = False  # 用于线程间通信，指示是否检测到障碍物
current_orientation = 2 # 初始化小车朝向为下
color_is_red=False
flags=True


# 定义摩斯电码的时间单位
DOT_DURATION = 0.1
DASH_DURATION = 0.3
SPACE_DURATION = 0.1
LETTER_SPACE = 0.3
WORD_SPACE = 0.7

# 定义摩斯电码字典
MORSE_CODE_DICT = {
    'A': '.-', 'B': '-...', 'C': '-.-.', 'D': '-..', 'E': '.', 'F': '..-.',
    'G': '--.', 'H': '....', 'I': '..', 'J': '.---', 'K': '-.-', 'L': '.-..',
    'M': '--', 'N': '-.', 'O': '---', 'P': '.--.', 'Q': '--.-', 'R': '.-.',
    'S': '...', 'T': '-', 'U': '..-', 'V': '...-', 'W': '.--', 'X': '-..-',
    'Y': '-.--', 'Z': '--..', '1': '.----', '2': '..---', '3': '...--',
    '4': '....-', '5': '.....', '6': '-....', '7': '--...', '8': '---..',
    '9': '----.', '0': '-----', ',': '--..--', '.': '.-.-.-', '?': '..--..',
    '/': '-..-.', '-': '-....-', '(': '-.--.', ')': '-.--.-', ' ': ' '
}

def set_led_color(r, g, b):
    GPIO.output(LED_R, r)
    GPIO.output(LED_G, g)
    GPIO.output(LED_B, b)

def beep_and_light(duration, color):
    global pwm_BUZZER_PIN 
    pwm_BUZZER_PIN.start(50)  # 50% 占空比
    set_led_color(*color)  # 设置灯光颜色
    time.sleep(duration)
    pwm_BUZZER_PIN.stop()
    set_led_color(GPIO.LOW, GPIO.LOW, GPIO.LOW)  # 关闭灯光
    time.sleep(SPACE_DURATION)

def play_morse_char(char):
    if char == ' ':
        time.sleep(WORD_SPACE - LETTER_SPACE)  # 减去已经在字母间等待的时间
        return
    
    morse = MORSE_CODE_DICT.get(char.upper(), '')
    for signal in morse:
        if signal == '.':
            beep_and_light(DOT_DURATION, (GPIO.HIGH, GPIO.HIGH, GPIO.LOW))  # 黄色灯光
        elif signal == '-':
            beep_and_light(DASH_DURATION, (GPIO.HIGH, GPIO.LOW, GPIO.LOW))  # 红色灯光
    time.sleep(LETTER_SPACE - SPACE_DURATION)  # 减去已经在beep中等待的时间

def play_morse_string(text):
    global pwm_BUZZER_PIN
    pwm_BUZZER_PIN = GPIO.PWM(BuzzerPin, 440)
    for char in text:
        play_morse_char(char)
    # 确保在字符串播放完毕后蜂鸣器停止，灯光关闭
    pwm_BUZZER_PIN.stop()
    set_led_color(GPIO.LOW, GPIO.LOW, GPIO.LOW)
    pwm_BUZZER_PIN=None

def text_to_morse(text):
    return ' '.join(MORSE_CODE_DICT.get(char.upper(), '') for char in text)

def play_morse_code(text):
    """
    播放文本的摩斯电码。

    Args:
        text: 要播放的文本。
    """
    morse_code = text_to_morse(text)
    print(f"摩斯码: {morse_code}")
    play_morse_string(text)
    print("播放完成")

def servo_pulse(myangle,myPin):
    pulsewidth = (myangle * 11) + 500
    GPIO.output(myPin, GPIO.HIGH)
    time.sleep(pulsewidth/1000000.0)
    GPIO.output(myPin, GPIO.LOW)
    time.sleep(20.0/1000-pulsewidth/1000000.0)

def my_control_servo(angle, my_servo_pin):
    """
    控制舵机转动到指定角度。
    参数：
    angle: 舵机目标角度，单位为度。
    my_servo_pin: 舵机连接的 GPIO 引脚。
    """
    global pwm_servo_USPin, pwm_servo_cameraV, pwm_servo_cameraH

    # 初始化 GPIO
    GPIO.setmode(GPIO.BCM)

    if my_servo_pin == USPin:
        if pwm_servo_USPin is None:
            GPIO.setup(USPin, GPIO.OUT)
            pwm_servo_USPin = GPIO.PWM(USPin, 50)  # 50Hz
            pwm_servo_USPin.start(0)
        pwm = pwm_servo_USPin
    elif my_servo_pin == cameraV:
        if pwm_servo_cameraV is None:
            GPIO.setup(cameraV, GPIO.OUT)
            pwm_servo_cameraV = GPIO.PWM(cameraV, 50)  # 50Hz
            pwm_servo_cameraV.start(0)
        pwm = pwm_servo_cameraV
    elif my_servo_pin == cameraH:
        if pwm_servo_cameraH is None:
            GPIO.setup(cameraH, GPIO.OUT)
            pwm_servo_cameraH = GPIO.PWM(cameraH, 50)  # 50Hz
            pwm_servo_cameraH.start(0)
        pwm = pwm_servo_cameraH
    else:
        raise ValueError(f"无效的舵机名称: {my_servo_pin}")

    # 假设舵机工作范围为 0° 到 180°，占空比范围为 2.5% 到 12.5%
    duty_cycle = (angle / 180) * 10 + 2.5  # 计算占空比
    pwm.ChangeDutyCycle(duty_cycle)
    time.sleep(0.5)  # 等待 0.5 秒
    pwm.ChangeDutyCycle(0)  # 停止 PWM 输出信号

    # 清理引脚
    cleanup(my_servo_pin)

def cleanup(my_servo_pin):
    """
    停止指定 GPIO 引脚上的 PWM 输出，并清理该引脚。
    参数：
    my_servo_pin: 需要清理的 GPIO 引脚。
    """
    global pwm_servo_USPin, pwm_servo_cameraV, pwm_servo_cameraH

    if my_servo_pin == USPin and pwm_servo_USPin is not None:
        pwm_servo_USPin.stop()
        GPIO.cleanup(USPin)
        pwm_servo_USPin = None
    elif my_servo_pin == cameraV and pwm_servo_cameraV is not None:
        pwm_servo_cameraV.stop()
        GPIO.cleanup(cameraV)
        pwm_servo_cameraV = None
    elif my_servo_pin == cameraH and pwm_servo_cameraH is not None:
        pwm_servo_cameraH.stop()
        GPIO.cleanup(cameraH)
        pwm_servo_cameraH = None
    else:
        raise ValueError(f"无效的舵机名称或舵机未初始化: {my_servo_pin}")


                        
#舵机来回转动 myangle 1-181
def servo_control(myangle,myPin):
    for pos in range(myangle):
        servo_pulse(pos,myPin)
        time.sleep(0.005) 
    time.sleep(1)
    for pos in reversed(range(myangle)):
        servo_pulse(pos,myPin)
        time.sleep(0.005)
    time.sleep(1)
    
def servo_control_90(myPin):
    for pos in range(90):
        servo_pulse(pos,myPin)
        time.sleep(0.005) 
    time.sleep(1)
#调整舵机旋转
def servo_control_fin(myangle,myPin): 
    servo_control(myangle,myPin)
    servo_control_90(myPin)

class Node:
    def __init__(self, position, parent=None):
        self.position = position  # 格子的坐标，如(0, 0)
        self.parent = parent  # 父节点
        self.g = 0  # 从起点到该节点的移动代价
        self.h = 0  # 启发式代价（从该节点到终点的估计代价）
        self.f = 0  # 总代价（g + h）

    def __lt__(self, other):
        return self.f < other.f

def heuristic(a, b):
    # 使用曼哈顿距离作为启发式函数
    return abs(a[0] - b[0]) + abs(a[1] - b[1])

def astar(grid, start, end):
    open_list = []  # 待探索节点列表
    closed_list = set()  # 已探索节点列表
    
    start_node = Node(start)  # 创建起点节点
    end_node = Node(end)  # 创建终点节点
    
    heapq.heappush(open_list, start_node)  # 将起点节点加入开放列表
    
    while open_list:  # 当开放列表不为空时
        current_node = heapq.heappop(open_list)  # 从开放列表中取出 f 值最小的节点
        closed_list.add(current_node.position)  # 将当前节点加入关闭列表
        
        # 找到终点，返回路径
        if current_node.position == end_node.position:
            path = []  # 初始化路径列表
            while current_node:  # 从当前节点回溯到起点
                path.append(current_node.position)  # 将节点坐标加入路径列表
                current_node = current_node.parent  # 获取父节点
            return path[::-1]  # 反转路径，得到从起点到终点的路径
        
        # 获取邻居节点
        neighbors = [
            (0, -1), (-1, 0), (0, 1), (1, 0)  # 上下左右四个方向
        ]
        for dx, dy in neighbors:  # 遍历每个邻居方向
            neighbor_pos = (current_node.position[0] + dx, current_node.position[1] + dy)  # 计算邻居节点坐标
            
            # 检查邻居节点是否在网格内且不是障碍物
            if 0 <= neighbor_pos[0] < len(grid) and 0 <= neighbor_pos[1] < len(grid[0]):
                if neighbor_pos in closed_list or grid[neighbor_pos[0]][neighbor_pos[1]] == 'X':
                    continue  # 如果邻居节点在关闭列表中或为障碍物，则跳过
                
                neighbor_node = Node(neighbor_pos, current_node)  # 创建邻居节点
                neighbor_node.g = current_node.g + 1  # 计算邻居节点的 g 值
                neighbor_node.h = heuristic(neighbor_pos, end_node.position)  # 计算邻居节点的 h 值
                neighbor_node.f = neighbor_node.g + neighbor_node.h  # 计算邻居节点的 f 值
                
                # 检查邻居节点是否已经在开放列表中，如果在，比较 g 值，选择更小的 g 值
                if not any(node.position == neighbor_node.position and node.g <= neighbor_node.g for node in open_list):
                    heapq.heappush(open_list, neighbor_node)  # 将邻居节点加入开放列表
    
    return None  # 未找到路径
# 定义5x5方格，'X'表示障碍物
# grid = [
#     ['A', 'A', 'X', 'X', 'A'],
#     ['A', 'A', 'A', 'A', 'A'],
#     ['X', 'A', 'X', 'A', 'A'],
#     ['A', 'A', 'A', 'A', 'A'],
#     ['A', 'A', 'A', 'A', 'A']
# ]
grid = [
    ['A', 'A', 'A', 'A', 'A'],
    ['A', 'A', 'A', 'X', 'A'],
    ['A', 'A', 'X', 'A', 'A'],
    ['A', 'A', 'A', 'A', 'A'],
    ['A', 'A', 'X', 'A', 'A']
]
def grid_to_string(grid):
    result = ""
    for row in grid:
        result += ' '.join(row) + '\n'
    return result

# 获取用户输入，决定是否执行 panorama()
def get_panorama_choice():
    while True:
        try:
            choice = input("是否执行全景拍摄 (y/n)? ").strip().lower()
            if choice in ('y', 'n'):
                return choice == 'y'
            else:
                print("输入无效。请输入 'y' 或 'n'。")
        except Exception as e:
            print(f"输入错误: {e}")

# 获取用户输入的起点和终点
def get_input(prompt):
    global flags
    while True:
        try:
            value = input(prompt).strip().upper()
            if(value=="exit"):
                flags=False
                return "exit"
            if len(value) == 2 and value[0] in "ABCDE" and value[1] in "12345":
                row = ord(value[0]) - ord('A')
                col = int(value[1]) - 1
                if grid[row][col] != 'X':
                    return (row, col)
                else:
                    print("位置是障碍物，请选择其他位置。")
            else:
                print("输入无效。请使用格式如A1、B2等。")
        except Exception as e:
            print(f"输入错误: {e}")

def format_path(path):
    formatted_path = []
    for pos in path:
        row_char = chr(pos[0] + ord('A'))
        col_char = str(pos[1] + 1)
        formatted_path.append(f"{row_char}{col_char}")
    return formatted_path

def init():
    global pwm_ENA, pwm_ENB, pwm_servo_cameraH,pwm_servo_cameraV,pwm_servo_USPin
    GPIO.setmode(GPIO.BCM)
    GPIO.setup(ENA, GPIO.OUT, initial=GPIO.HIGH)
    GPIO.setup(IN1, GPIO.OUT, initial=GPIO.LOW)
    GPIO.setup(IN2, GPIO.OUT, initial=GPIO.LOW)
    GPIO.setup(ENB, GPIO.OUT, initial=GPIO.HIGH)
    GPIO.setup(IN3, GPIO.OUT, initial=GPIO.LOW)
    GPIO.setup(IN4, GPIO.OUT, initial=GPIO.LOW)
    GPIO.setup(key, GPIO.IN)
    GPIO.setup(EchoPin, GPIO.IN)
    GPIO.setup(TrigPin, GPIO.OUT)
    GPIO.setup(LED_R, GPIO.OUT)
    GPIO.setup(LED_G, GPIO.OUT)
    GPIO.setup(LED_B, GPIO.OUT)
    GPIO.setup(USPin, GPIO.OUT)
    GPIO.setup(cameraV, GPIO.OUT)
    GPIO.setup(cameraH, GPIO.OUT)
    GPIO.setup(TrackSensorLeftPin1, GPIO.IN)
    GPIO.setup(TrackSensorLeftPin2, GPIO.IN)
    GPIO.setup(TrackSensorRightPin1, GPIO.IN)
    GPIO.setup(TrackSensorRightPin2, GPIO.IN)
    GPIO.setup(BuzzerPin, GPIO.OUT)
    # 设置pwm引脚和频率为2000hz
    # pwm_BUZZER_PIN = GPIO.PWM(BuzzerPin, 440)  # 440Hz 是一个容易听到的频率
    pwm_ENA = GPIO.PWM(ENA, 2000)
    pwm_ENB = GPIO.PWM(ENB, 2000)
    pwm_ENA.start(0)
    pwm_ENB.start(0)
    # 设置舵机的频率和起始占空比
    pwm_servo_USPin = GPIO.PWM(USPin, 50)
    pwm_servo_cameraV = GPIO.PWM(cameraV, 50)
    pwm_servo_cameraH = GPIO.PWM(cameraH, 50)
    # pwm_servo_USPin.start(0)
    pwm_servo_cameraH.start(0)
    pwm_servo_cameraV.start(0)

def my_end():
    """
    清理 GPIO 引脚和 PWM 对象。
    """
    global pwm_ENA, pwm_ENB, pwm_servo_cameraH, pwm_servo_cameraV, pwm_servo_USPin
    
    # 停止 PWM 输出
    if pwm_ENA is not None:
        pwm_ENA.stop()
    if pwm_ENB is not None:
        pwm_ENB.stop()
    if pwm_servo_USPin is not None:
        pwm_servo_USPin.stop()
    if pwm_servo_cameraH is not None:
        pwm_servo_cameraH.stop()
    if pwm_servo_cameraV is not None:
        pwm_servo_cameraV.stop()
    if pwm_BUZZER_PIN is not None:
        pwm_BUZZER_PIN.stop()
    # 清理 GPIO 引脚
    GPIO.cleanup()

# 小车前进
def run(leftspeed, rightspeed):
    GPIO.output(IN1, GPIO.HIGH)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.HIGH)
    GPIO.output(IN4, GPIO.LOW)
    pwm_ENA.ChangeDutyCycle(leftspeed)
    pwm_ENB.ChangeDutyCycle(rightspeed)
    # 设置前进时的LED灯颜色为绿色
    GPIO.output(LED_R, GPIO.LOW)
    GPIO.output(LED_G, GPIO.HIGH)
    GPIO.output(LED_B, GPIO.LOW)

# 小车后退
def back(leftspeed, rightspeed):
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.HIGH)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.HIGH)
    pwm_ENA.ChangeDutyCycle(leftspeed)
    pwm_ENB.ChangeDutyCycle(rightspeed)
    # 设置后退时的LED灯颜色为绿色
    GPIO.output(LED_R, GPIO.LOW)
    GPIO.output(LED_G, GPIO.HIGH)
    GPIO.output(LED_B, GPIO.LOW)

# 小车左转
def left(leftspeed, rightspeed):
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.HIGH)
    GPIO.output(IN4, GPIO.LOW)
    pwm_ENA.ChangeDutyCycle(leftspeed)
    pwm_ENB.ChangeDutyCycle(rightspeed)
    # 设置左转时的LED灯颜色为蓝色
    GPIO.output(LED_R, GPIO.LOW)
    GPIO.output(LED_G, GPIO.LOW)
    GPIO.output(LED_B, GPIO.HIGH)

# 小车右转
def right(leftspeed, rightspeed):
    GPIO.output(IN1, GPIO.HIGH)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.LOW)
    pwm_ENA.ChangeDutyCycle(leftspeed)
    pwm_ENB.ChangeDutyCycle(rightspeed)
    # 设置右转时的LED灯颜色为蓝色
    GPIO.output(LED_R, GPIO.LOW)
    GPIO.output(LED_G, GPIO.LOW)
    GPIO.output(LED_B, GPIO.HIGH)

# 小车原地左转
def spin_left(leftspeed, rightspeed):
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.HIGH)
    GPIO.output(IN3, GPIO.HIGH)
    GPIO.output(IN4, GPIO.LOW)
    pwm_ENA.ChangeDutyCycle(leftspeed)
    pwm_ENB.ChangeDutyCycle(rightspeed)
    # 设置原地左转时的LED灯颜色为红色
    GPIO.output(LED_R, GPIO.HIGH)
    GPIO.output(LED_G, GPIO.LOW)
    GPIO.output(LED_B, GPIO.LOW)

# 小车原地右转
def spin_right(leftspeed, rightspeed):
    GPIO.output(IN1, GPIO.HIGH)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.HIGH)
    pwm_ENA.ChangeDutyCycle(leftspeed)
    pwm_ENB.ChangeDutyCycle(rightspeed)
    # 设置原地右转时的LED灯颜色为红色
    GPIO.output(LED_R, GPIO.HIGH)
    GPIO.output(LED_G, GPIO.LOW)
    GPIO.output(LED_B, GPIO.LOW)

# 小车停止
def brake():
    GPIO.output(IN1, GPIO.LOW)
    GPIO.output(IN2, GPIO.LOW)
    GPIO.output(IN3, GPIO.LOW)
    GPIO.output(IN4, GPIO.LOW)
    # 设置停止时的LED灯颜色熄灭
    GPIO.output(LED_R, GPIO.LOW)
    GPIO.output(LED_G, GPIO.LOW)
    GPIO.output(LED_B, GPIO.LOW)

# 按键检测
def key_scan():
    while GPIO.input(key):
        pass
    while not GPIO.input(key):
        time.sleep(0.01)
        if not GPIO.input(key):
            time.sleep(0.01)
            while not GPIO.input(key):
                pass

# 判断是否在节点上
def track_node_check():
    TrackSensorLeftValue1 = GPIO.input(TrackSensorLeftPin1)
    TrackSensorLeftValue2 = GPIO.input(TrackSensorLeftPin2)
    TrackSensorRightValue1 = GPIO.input(TrackSensorRightPin1)
    TrackSensorRightValue2 = GPIO.input(TrackSensorRightPin2)
    return TrackSensorLeftValue1 == False and TrackSensorLeftValue2 == False and TrackSensorRightValue1 == False and TrackSensorRightValue2 == False

# 判断是否在节点上
def is_at_node():
    return track_node_check()

# 超声波测距函数
def Distance():
    GPIO.output(TrigPin, GPIO.LOW)
    time.sleep(0.000002)
    GPIO.output(TrigPin, GPIO.HIGH)
    time.sleep(0.000015)
    GPIO.output(TrigPin, GPIO.LOW)

    t3 = time.time()
    while not GPIO.input(EchoPin):
        t4 = time.time()
        if (t4 - t3) > 0.10:
            return -1

    t1 = time.time()
    while GPIO.input(EchoPin):
        t5 = time.time()
        if (t5 - t1) > 0.10:
            return -1

    t2 = time.time()
    time.sleep(0.01)
    return ((t2 - t1) * 340 / 2) * 100

def Distance_test():
    num = 0
    ultrasonic = []
    while num < 4:
        distance = Distance()
        while int(distance) == -1:
            distance = Distance()
        while int(distance) >= 500 or int(distance) == 0:
            distance = Distance()
        ultrasonic.append(distance)
        num += 1
        time.sleep(0.01)
    distance = (ultrasonic[1] + ultrasonic[2] + ultrasonic[3]) / 3
    # print("distance is %f"%(distance))
    return distance

# 超声波避障
def avoid_obstacle():
    while True:
        global obstacle_detected
        distance = Distance_test()
        if distance < 18:  # 检测到24厘米以内的障碍物
            obstacle_detected = True
        else:
            obstacle_detected = False

# 循迹函数
def run_track():
    global obstacle_detected
    global color_is_red
    global current_orientation 
    run(20,20)
    time.sleep(1)
    
    while not track_node_check():
        if obstacle_detected == True:
            print("检测到障碍物，停止并重新规划路径")
            brake()
            time.sleep(1)
            return False
        
#         # 判断红绿灯  
#         if color_is_red:  
#             print("红灯，等待...")
#             brake()  
#             time.sleep(1)
#             while color_is_red:  # 等待直到红灯变为绿灯  
#                 time.sleep(0.1)  # 短暂休眠以避免过度占用CPU  
#             print("绿灯，前进")

        # 检测到黑线时循迹模块相应的指示灯亮，端口电平为LOW
        # 未检测到黑线时循迹模块相应的指示灯灭，端口电平为HIGH
        TrackSensorLeftValue1 = GPIO.input(TrackSensorLeftPin1)
        TrackSensorLeftValue2 = GPIO.input(TrackSensorLeftPin2)
        TrackSensorRightValue1 = GPIO.input(TrackSensorRightPin1)
        TrackSensorRightValue2 = GPIO.input(TrackSensorRightPin2)
        
        # 四路循迹引脚电平状态
        # X 0 1 X
        # 处理左小弯
        if TrackSensorLeftValue2 == False and TrackSensorRightValue1 == True or TrackSensorLeftValue1 ==False:
            left(0, 60)

        # 四路循迹引脚电平状态
        # X 1 0 X  
        # 处理右小弯
        elif TrackSensorLeftValue2 == True and TrackSensorRightValue1 == False or TrackSensorRightValue2 == False:
            right(60, 0)

        # 四路循迹引脚电平状态
        # X 0 0 X
        # 处理直线
        elif TrackSensorLeftValue2 == False and TrackSensorRightValue1 == False:
            run(20,20)
        
        else:
            spin_left(25,25)#5 5
        
    return True

     # 根据当前朝向确定小车的左右转向

# 循迹返回函数
def run_track_back():
    while not track_node_check():
        # 检测到黑线时循迹模块相应的指示灯亮，端口电平为LOW
        # 未检测到黑线时循迹模块相应的指示灯灭，端口电平为HIGH
        TrackSensorLeftValue1 = GPIO.input(TrackSensorLeftPin1)
        TrackSensorLeftValue2 = GPIO.input(TrackSensorLeftPin2)
        TrackSensorRightValue1 = GPIO.input(TrackSensorRightPin1)
        TrackSensorRightValue2 = GPIO.input(TrackSensorRightPin2)
        
        # 四路循迹引脚电平状态
        # X 0 1 X
        # 处理左小弯
        if TrackSensorLeftValue2 == False and TrackSensorRightValue1 == True or TrackSensorLeftValue1 ==False:
            left(0, 80)
        
        # 四路循迹引脚电平状态
        # X 1 0 X  
        # 处理右小弯
        elif TrackSensorLeftValue2 == True and TrackSensorRightValue1 == False or TrackSensorRightValue2 == False:
            right(80, 0)

        # 四路循迹引脚电平状态
        # X 0 0 X
        # 处理直线
        elif TrackSensorLeftValue2 == False and TrackSensorRightValue1 == False:
            run(10,10)
        
        else:
            spin_left(20,20)

# 循迹函数
def run_track():
    global obstacle_detected
    global color_is_red
    global current_orientation 
    run(10,10)
    time.sleep(0.4)
    
    while not track_node_check():
        if obstacle_detected == True:
            print("检测到障碍物，停止并重新规划路径")
            brake()
            time.sleep(1)
            return False
        
        # 判断红绿灯  
        # if color_is_red:  
        #     print("红灯，等待...")
        #     brake()  
        #     time.sleep(1)
        #     while color_is_red:  # 等待直到红灯变为绿灯  
        #         time.sleep(0.1)  # 短暂休眠以避免过度占用CPU  
        #     print("绿灯，前进")

        # 检测到黑线时循迹模块相应的指示灯亮，端口电平为LOW
        # 未检测到黑线时循迹模块相应的指示灯灭，端口电平为HIGH
        TrackSensorLeftValue1 = GPIO.input(TrackSensorLeftPin1)
        TrackSensorLeftValue2 = GPIO.input(TrackSensorLeftPin2)
        TrackSensorRightValue1 = GPIO.input(TrackSensorRightPin1)
        TrackSensorRightValue2 = GPIO.input(TrackSensorRightPin2)
        
        # 四路循迹引脚电平状态
        # X 0 1 X
        # 处理左小弯
        if TrackSensorLeftValue2 == False and TrackSensorRightValue1 == True or TrackSensorLeftValue1 ==False:
            left(0, 40)

        # 四路循迹引脚电平状态
        # X 1 0 X  
        # 处理右小弯
        elif TrackSensorLeftValue2 == True and TrackSensorRightValue1 == False or TrackSensorRightValue2 == False:
            right(40, 0)

        # 四路循迹引脚电平状态
        # X 0 0 X
        # 处理直线
        elif TrackSensorLeftValue2 == False and TrackSensorRightValue1 == False:
            run(10,10)
        
        else:
            spin_left(20,20)
        
    return True

     # 根据当前朝向确定小车的左右转向
def update_orientation(next_orientation):
    global current_orientation
    if current_orientation == 1:
        if next_orientation == 2:
            spin_left(18,18)  # 后转
            time.sleep(1.2)
        elif next_orientation == 3:
            spin_left(15,15)  # 左转
            time.sleep(0.5)
        elif next_orientation == 4:
            spin_right(15,15)   # 右转
            time.sleep(0.8)
    elif current_orientation == 2:
        if next_orientation == 1:
            spin_left(18,18)  # 后转
            time.sleep(1.3)
        elif next_orientation == 3:
            spin_right(15,15)  # 左转
            time.sleep(0.8)
        elif next_orientation == 4:
            spin_left(15,15)   # 右转
            time.sleep(0.5)
    elif current_orientation == 3:
        if next_orientation == 1:
            spin_right(15,15)  # 右转
            time.sleep(0.80)
        elif next_orientation == 2:
            spin_left(15,15)  # 左转
            time.sleep(0.5)
        elif next_orientation == 4:
            spin_left(18,18)   # 后转 18 18
            time.sleep(1.2)
    elif current_orientation == 4:
        if next_orientation == 1:
            spin_left(15,15)  # 左转 15 15
            time.sleep(0.50)
        elif next_orientation == 2:
            spin_right(15,15)  # 右转
            time.sleep(0.8)
        elif next_orientation == 3:
            spin_left(18,18)   # 后转
            time.sleep(1.2)

# 移动到下一个位置
def move_to_next_position(current_pos, next_pos):
    global current_orientation
    global obstacle_detected
    
    # 确定下一个朝向
    if next_pos[0] == current_pos[0] - 1:  # 上
        next_orientation = 1
    elif next_pos[0] == current_pos[0] + 1:  # 下
        next_orientation = 2
    elif next_pos[1] == current_pos[1] - 1:  # 左
        next_orientation = 3
    elif next_pos[1] == current_pos[1] + 1:  # 右
        next_orientation = 4
    else:
        print("未找到路径")
    
    print("current_pos:",current_pos)
    print("next_pos:",next_pos)
    print("current_orientation:",current_orientation)
    print("next_orientation:",next_orientation)
    
    # 更新小车朝向
    update_orientation(next_orientation)
    current_orientation = next_orientation
    t = current_orientation

    # 移动小车到下一个节点
    if not run_track():
        if t == 1:
            t = 2
        elif t == 2:
            t = 1
        elif t == 3:
            t = 4
        elif t == 4:
            t = 3
        current_orientation=t
        spin_left(15,15)   # 后转 20 20
        time.sleep(1.2)
        run_track_back()
        brake()  # 再次到达节点后停止并等待一秒
        time.sleep(1)
        return False

    brake()  # 再次到达节点后停止并等待一秒
    time.sleep(1)
    return True

# 跟随路径移动
def follow_path(path):
    global current_orientation
    for i in range(len(path) - 1):
        if not move_to_next_position(path[i], path[i + 1]):
            grid[path[i+1][0]][path[i+1][1]] = 'X'
            SMAsending() 
            return path[i]
    return 

def SMAsending():
    # 创建邮件对象
    msg = MIMEMultipart()
    msg['From'] = '2196627306@qq.com'
    msg['To'] = '2196627306@qq.com'
    msg['Subject'] = '遇到未知障碍！地图已更新并发送。'
    str = grid_to_string(grid)

    # 邮件正文内容
    body = str
    msg.attach(MIMEText(body, 'plain'))

    # SMTP服务器设置
    server = smtplib.SMTP_SSL('smtp.qq.com', 465)  # 使用SSL
    server.login('2196627306@qq.com', 'aduepedzebuoeagj')  # 登录认证

    # 发送邮件
    server.sendmail(msg['From'], msg['To'], msg.as_string())
    server.quit()

def color_recongnition():
    global color_is_red
    # 初始化摄像头
    cap = cv2.VideoCapture(1)  # 1表示第一个摄像头设备

    min_area=1500
    max_area=5000
    
    if not cap.isOpened():
        print("无法打开摄像头")
    
    while True:
        # 读取当前帧
        ret, frame = cap.read()
        # 如果不能读取帧，退出循环
        if not ret:
            print("failed")
            break
        # 将图像转换为HSV颜色空间
        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        # 定义绿色的HSV阈值范围
        lower_green = np.array([50, 150, 150])
        upper_green = np.array([70, 255, 255])
        # 根据阈值创建绿色的掩码
        mask_green = cv2.inRange(hsv, lower_green, upper_green)
        # 查找绿色区域的轮廓
        contours_green, _ = cv2.findContours(mask_green, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # 如果检测到绿色的轮廓则为假
        if contours_green:
            color_is_red=False
            print('green')
        # 定义红色的HSV阈值范围（红色通常分两个区域）
        lower_red1 = np.array([0, 150, 150])
        upper_red1 = np.array([10, 255, 255])
        lower_red2 = np.array([170, 150, 150])
        upper_red2 = np.array([180, 255, 255])
        # 根据阈值创建红色的掩码
        mask_red1 = cv2.inRange(hsv, lower_red1, upper_red1)
        mask_red2 = cv2.inRange(hsv, lower_red2, upper_red2)
        mask_red = cv2.add(mask_red1, mask_red2)
        # 查找红色区域的轮廓
        contours_red, _ = cv2.findContours(mask_red, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        for contour in contours_red:
            area = cv2.contourArea(contour)
            if min_area<=area<=max_area:
                color_is_red=True
                print('red')

    # 释放摄像头并关闭所有窗口
    cap.release()
    cv2.destroyAllWindows()


def photo():
    my_control_servo(130,cameraV)
    my_control_servo(90,cameraH)
    # 初始化摄像头
    cap = cv2.VideoCapture(1)

    # 加载人脸和眼睛的haar级联分类器
    face_cascade = cv2.CascadeClassifier("haarcascade_frontalface_default.xml")

    # 检测到的人脸的坐标
    face_coords = None
    stable_frames = 0
    required_stable_frames = 10  # 设置需要人脸稳定的帧数

    while True:
        ret, frame = cap.read()
        if not ret:
            print("无法获取摄像头帧")
            break

        # 转换为灰度图像
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))

        if len(faces) > 0:
            x, y, w, h = faces[0]  # 取第一个检测到的人脸
            new_face_coords = (x, y, w, h)

            if face_coords is None:
                face_coords = new_face_coords
                stable_frames = 1
            else:
                # 如果人脸坐标变化不大，则认为人脸稳定
                if abs(face_coords[0] - new_face_coords[0]) < 10 and abs(face_coords[1] - new_face_coords[1]) < 10:
                    stable_frames += 1
                else:
                    stable_frames = 1
                face_coords = new_face_coords

            # 如果人脸稳定帧数达到了要求，拍摄照片并直接发送
            if stable_frames >= required_stable_frames:
                # 拍摄照片，保存原始帧
                image_path = "/home/pi/captured_image.jpg"
                cv2.imwrite(image_path, frame)  # 保存整个帧
                print(f"已保存照片: {image_path}")
                send_face_image()
                # 拍摄完照片后，关闭摄像头和画面
                cap.release()
                cv2.destroyAllWindows()
                break

        # 显示图像
        cv2.imshow('Video', frame)

        # 按 'q' 键退出
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 释放摄像头资源
    cap.release()
    cv2.destroyAllWindows()
    return True
    
#发送人脸照片
def send_face_image():
    host="192.168.23.154"
    port=6666
    image_path="/home/pi/captured_image.jpg"
    # 创建socket对象
    client_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    try:
        # 连接服务器
        client_socket.connect((host, port))
        print(f"已连接到 {host}:{port}")

        # 打开并读取图片文件
        with open(image_path, 'rb') as file:
            image_data = file.read()
        
        # 发送图片文件大小
        file_size = len(image_data)
        client_socket.send(str(file_size).encode())
        
        # 等待服务器确认
        client_socket.recv(1024)
        
        # 发送图片数据
        client_socket.sendall(image_data)
        
        print("图片发送成功")
        
    except Exception as e:
        print(f"发生错误: {e}")
    
    finally:
        client_socket.close()


  
def capture_image(camera,i):
    for _ in range(10): 
        # 多次读取，丢弃可能的缓存帧 
        ret, frame = camera.read() 
        time.sleep(0.001)
    time.sleep(1)
    ret, frame = camera.read()
    if ret:
        image_path = "image"+str(i)+".jpg"
        cv2.imwrite(image_path, frame)
        print("photo")
        return frame
    return None

def rotate_servo(myangle,myPin):
    print("逆时针"+str(myangle)+str(myPin))
    for pos in range(myangle):
            servo_pulse(pos,myPin)
            time.sleep(0.0001)
        
def rotate_cameraV(myangle):
    print("逆时针"+str(myangle))
    for pos in range(myangle):
            servo_pulse(pos,cameraV)
            time.sleep(0.0001)

def rotate_cameraH(myangle):
    print("逆时针"+str(myangle))
    for pos in range(myangle):
            servo_pulse(pos,cameraH)
            time.sleep(0.0001)
    
def send_image(sock, image_path):
    filename = os.path.basename(image_path)
    with open(image_path, 'rb') as file:
        image_data = file.read()
    # 发送文件名长度
    sock.sendall(struct.pack('>I', len(filename)))
    # 发送文件名
    sock.sendall(filename.encode('utf-8'))
    # 发送图片大小
    sock.sendall(struct.pack('>I', len(image_data)))
    # 发送图片数据
    sock.sendall(image_data)
#发送全景照片
def SendImages(ip, image_folder):
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.connect((ip, 16666))
        print(f"成功连接到服务器 {ip}:16666")

        # 获取文件夹中的所有jpg图片
        image_files = [f for f in os.listdir(image_folder) if f.endswith('.jpg')]
        print(f"找到 {len(image_files)} 张jpg图片")

        # 发送图片数量
        sock.sendall(struct.pack('>I', len(image_files)))

        for image_file in image_files:
            image_path = os.path.join(image_folder, image_file)
            send_image(sock, image_path)
            print(f"照片 {image_file} 已发送")

        print("所有照片已发送完毕")
        
    except socket.error as e:
        print(f"套接字错误: {e}")
    finally:
        sock.close()
        print("连接已关闭")
   
def stitch_images(images):
    # stitcher = cv2.Stitcher_create()
    # status, panorama = stitcher.stitch(images)
    stitcher = cv2.Stitcher_create(cv2.Stitcher_PANORAMA) 
    stitcher.setPanoConfidenceThresh(0.01) # 降低置信度阈值 
    status, panorama = stitcher.stitch(images)
    if status == cv2.Stitcher_OK:
        print("全景图拼接成功")
        return panorama
    else:
        print("全景图拼接失败")
        return None

def panorama():
    my_control_servo(135,cameraV)
    my_control_servo(90,cameraH)
    camera = cv2.VideoCapture(1)
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1280)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 960)

    if not camera.isOpened():
        print("无法打开摄像头")
        return
    time.sleep(2)  # 给摄像头预热的时间
    images = []
    for i in range(31):
        my_control_servo(1+i*6,cameraH)
        images.append(capture_image(camera,i))
        
    camera.release()
    # #发送图片
    SendImages("192.168.50.100","/home/pi/guidan")

In [3]:
def main_control_function():
    global current_orientation,flags
    # init()  # 初始化GPIO设置
    my_control_servo(90,USPin)
    if photo():
        print("人脸识别成功")
        print("欢迎使用智能小车控制系统！")
    else:
        print("人脸识别失败，退出")
        return 
    
    if not is_at_node():
        print("不在节点上")
        return
    
    obstacle_thread = threading.Thread(target=avoid_obstacle)
    obstacle_thread.start()
    print("请选择起点和终点：")
    start = get_input("请输入起点位置：")
    end = get_input("请输入终点位置：")
    while True:
        print(f"起点位置：{chr(start[0] + ord('A'))}{start[1] + 1}")
        print(f"终点位置：{chr(end[0] + ord('A'))}{end[1] + 1}")
        path = astar(grid, start, end)
        if path:
            print("找到路径：", format_path(path))
            obstacle_position = follow_path(path)
            if obstacle_position:
                # 更新路径
                path = astar(grid, obstacle_position, end)
                if not path:
                    print("无法找到绕过障碍物的路径")
                    start = obstacle_position
                    end = get_input("请输入下一终点位置：")
                else:
                    print("重新规划路径：")
                    start = obstacle_position
                    continue
            else:
                brake()  # 到达终点后停止
                time.sleep(1)
                # 获取用户输入，决定是否执行 panorama()
                text=input("输入转化为莫斯电码的文本)")
                play_morse_code(text)
                GPIO.output(BuzzerPin,GPIO.HIGH)
                panorama_choice = get_panorama_choice()
                if panorama_choice:
                    panorama()
                start = end
                end = get_input("请输入下一终点位置：")
        else:
            print("未找到路径。")
    obstacle_thread.join()
    
    
    
    

In [4]:

# 主程序入口
if __name__ == '__main__':
    try:
        init()
        my_control_servo(90,cameraV)
        my_control_servo(90,cameraH)
        my_control_servo(90,USPin)
        main_control_function()
    except KeyboardInterrupt:
        pass
    finally:
        my_end()
    


已保存照片: /home/pi/captured_image.jpg
已连接到 192.168.23.154:6666
图片发送成功
人脸识别成功
欢迎使用智能小车控制系统！
请选择起点和终点：
起点位置：A1
终点位置：E1
找到路径： ['A1', 'B1', 'C1', 'D1', 'E1']
current_pos: (0, 0)
next_pos: (1, 0)
current_orientation: 2
next_orientation: 2
current_pos: (1, 0)
next_pos: (2, 0)
current_orientation: 2
next_orientation: 2
检测到障碍物，停止并重新规划路径
重新规划路径：
起点位置：B1
终点位置：E1
找到路径： ['B1', 'B2', 'C2', 'D2', 'D1', 'E1']
current_pos: (1, 0)
next_pos: (1, 1)
current_orientation: 1
next_orientation: 4
current_pos: (1, 1)
next_pos: (2, 1)
current_orientation: 4
next_orientation: 2
current_pos: (2, 1)
next_pos: (3, 1)
current_orientation: 2
next_orientation: 2
current_pos: (3, 1)
next_pos: (3, 0)
current_orientation: 2
next_orientation: 3
current_pos: (3, 0)
next_pos: (4, 0)
current_orientation: 3
next_orientation: 2
摩斯码: ... --- ...
播放完成
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
photo
pho